In [ ]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch
import yaml
import scvi

In [ ]:
control_key = "is_control"
condition_keys = "target_gene"
condition_rep_keys = "gene_embeddings"
data_origin = "ArcVirtualCell_42_0.1_False_X_state_128" #{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}_{sample_rep_scaled}_{n_comps}
sample_rep = "X_state"
scvi_model_load_path = f"./data/processed/model/{sample_rep}_ncomps128_hidden2048_ArcVirtualCell_42_0.1_False"
state_model_load_path = f"./data/processed/model/X_state_ArcVirtualCell.pt"

experiment = "exp_20260202_150829_ArcVirtualCell_42_0.1_False_X_state_128_delta0.5_regm0.4"
experiment_load_path = os.path.join("experiments",experiment)
config_load_path = os.path.join(experiment_load_path, "config.yaml")
model_load_path = os.path.join(experiment_load_path, "checkpoints", "test_epoch_30000.pt") 
results_save_path = os.path.join("results",experiment)
os.makedirs(results_save_path, exist_ok=True)

In [ ]:
# load data
preprocess_save_path = f"./data/processed/{data_origin}"
adata_control = sc.read_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train = sc.read_h5ad(f"{preprocess_save_path}_train.h5ad")
if os.path.exists(f"{preprocess_save_path}_test.h5ad"):
    adata_test = sc.read_h5ad(f"{preprocess_save_path}_test.h5ad")
else:
    adata_test = None
print(adata_control)
print(adata_train)
print(adata_test)

In [ ]:
# load model
from src.training import FNet
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with open(config_load_path, 'r') as f:
    config = yaml.safe_load(f)
    print(f"load saved_config from {config_load_path}")
model = FNet(config["in_out_dim"], config["hidden_dim_v"], config["n_hiddens_v"], config["hidden_dim_g"], config["n_hiddens_g"], 
             config["condition_dim"], config["con_embedding_dim"], config["hidden_dim_con"], 
             config["time_dim"] ,config["time_embedding_dim"],config["hidden_dim_time"], 
             config["bottle_dim"], config["activation"])
checkpoint = torch.load(model_load_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

## inference & evalutaion

In [ ]:
train_conditions = list(adata_train.obs[condition_keys].unique())
test_conditions = list(adata_test.obs[condition_keys].unique())
print("train:")
print(train_conditions)
print("test:")
print(test_conditions)

In [ ]:
from src.evaluate import run_batch_inference
from src.evaluate import batch_reconstruct_pca, batch_reconstruct_scvi, batch_reconstruct_flatvi, batch_reconstruct_state
from src.evaluate.evals import *
from src.evaluate import evaluate_latent,evaluate_population_average,evaluate_population_distribution

In [ ]:
target_genes = ['CHMP3', 'AKT2', 'SHPRH']
# target_genes = ['CHMP3', 'AKT2', 'SHPRH', 'TMSB4X', 'KLF10', 'TARBP2', 'KDM2B']
#target_genes = ['CDCA2', 'DHX36', 'TMSB10', 'KAT2A', 'PHF14', 'KLF10', 'KDM1A']

# target_genes = ['CDCA2', 'DHX36', 'TMSB10']
                
n_particles = 5000
all_indices = np.arange(adata_control.n_obs)
rng = np.random.default_rng(42)
indices = rng.choice(all_indices, n_particles, replace=False) if n_particles < len(all_indices) else all_indices
adata_source = torch.tensor(adata_control.obsm[sample_rep][indices], dtype=torch.float32, device=device)

results_embedding = run_batch_inference(
    model=model,
    adata_source=adata_source,
    adata_conditions=adata_train, # 这里要用对应的adata 
    target_conditions=target_genes,
    condition_keys = condition_keys,
    embedding_key = condition_rep_keys,
    source_rep = sample_rep,
    n_steps = 50, # ode step
    device = device,
    random_seed = 42
)

In [ ]:
# 我们可以reconstruct出原始基因表达
if sample_rep == "X_pca_scaled":
    results_genes = batch_reconstruct_pca(
        inference_results=results_embedding,  
        ref_adata=adata_control        
    )
elif sample_rep in ["X_scVI"]:
    batch_data = adata_control.obs['_scvi_batch'].values[indices]
    model_ref = scvi.model.SCVI.load(f"{scvi_model_load_path}_ref", adata=adata_control)
    model_train = scvi.model.SCVI.load(f"{scvi_model_load_path}_train", adata=adata_control)
    model_test = scvi.model.SCVI.load(f"{scvi_model_load_path}_test", adata=adata_control)
    results_genes = batch_reconstruct_scvi(
        inference_results=results_embedding,
        scvi_model=model_ref,  # 传入模型
        target_library_size=1e4, # 输出将被标准化到 10,000 counts
        batch_idx = torch.full((n_particles, 1), 0, dtype=torch.long, device=device), #全部投射到第0个batch
        # batch_idx = torch.tensor(adata_control.obs['_scvi_batch'].values[indices], dtype=torch.long, device=device).unsqueeze(1) # 如果希望都投射到第0个batch 可以直接不传或者传None
    )
elif sample_rep =="X_flatvi":
    model_ref = scvi.model.SCVI.load(f"{scvi_model_load_path}_ref", adata=adata_control)
    results_genes = batch_reconstruct_flatvi(
        inference_results=results_embedding,
        flatvi_model=model_ref,
        target_library_size=1e4,
        var_names = adata_control.var_names
    )
elif sample_rep == "X_state":
    from src.preprocessing import NBDecoder, NBDecoderTrainer
    z_dim = adata_control.obsm["X_state"].shape[1]
    n_genes = adata_control.n_vars
    state_decoder = NBDecoder(z_dim=z_dim, n_genes=n_genes, hidden=(1024,2048,4096), dropout=0.1)
    trainer = NBDecoderTrainer(state_decoder, device="cuda", use_amp=False)
    trainer.load(state_model_load_path)  # 会自动把 state_dict 加载到 trainer.decoder 里
    state_decoder = trainer.decoder
    state_decoder.eval()
    
    results_genes = batch_reconstruct_state(
        inference_results=results_embedding,
        state_decoder=state_decoder,
        target_library_size=1e4,
        var_names = adata_control.var_names
    )

In [ ]:
#print(results_embedding)
#print(results_genes)

In [ ]:
if sample_rep in ["X_scVI","X_scVI_linear"]: 
    from src.evaluate import get_origin_expression
    get_origin_expression(adata_control,model_ref,target_library_size=1e4,sample_rep=sample_rep,batch_size=2048)
    get_origin_expression(adata_train,model_train,target_library_size=1e4,sample_rep=sample_rep,batch_size=2048)
    if adata_test is not None:
        get_origin_expression(adata_test,model_test,target_library_size=1e4,sample_rep=sample_rep,batch_size=2048)

    # sc.pp.normalize_total(adata_control, target_sum=1e4)
    # sc.pp.normalize_total(adata_train, target_sum=1e4)
    # sc.pp.normalize_total(adata_test, target_sum=1e4)

In [ ]:
latent_evaluate_results = evaluate_latent(
    results_embedding=results_embedding,
    adata_treated=adata_train,
    adata_control=adata_control,
    pert_key=condition_keys,        
    control_label=control_key,         
    embedding_key=sample_rep 
)
print(latent_evaluate_results)
latent_evaluate_results_save_path = os.path.join(results_save_path, "latent_metrics.csv")
latent_evaluate_results.to_csv(latent_evaluate_results_save_path, index=True) 

In [ ]:
average_evaluate_results = evaluate_population_average(
    results_genes=results_genes,
    adata_treated=adata_train,
    adata_control=adata_control,
    pert_key=condition_keys,        
    control_label=control_key,         
    embedding_key=sample_rep,
    Edistance_sample_num = 200,
    random_seed = 42
)
print(average_evaluate_results)
average_evaluate_results_save_path = os.path.join(results_save_path, "average_metrics.csv")
average_evaluate_results.to_csv(average_evaluate_results_save_path, index=True) 

In [ ]:
distribution_evaluate_results = evaluate_population_distribution(
    results_genes=results_genes,
    adata_treated=adata_train,
    adata_control=adata_control,
    pert_key=condition_keys,        
    max_cells=200,
    max_genes=20000, # 没有做hvg先算了吧
    n_bins=50,
    top_n_degs=50,
    seed=42,
)
print(distribution_evaluate_results)
distribution_evaluate_results_save_path = os.path.join(results_save_path, "distribution_metrics.csv")
distribution_evaluate_results.to_csv(distribution_evaluate_results_save_path, index=True) 

In [ ]:
# 运行主要评估 (MSE, R2, PCC, Spearman)
df_results = evaluate_all(
    results_genes=results_genes,
    results_embedding=results_embedding,
    adata_train=adata_train,
    adata_control=adata_control,
    pert_key=condition_keys,        
    control_label=control_key,         
    embedding_key=sample_rep 
)

print("评估结果预览：")
print(df_results)

pdisc_scores, pdisc_mean = evaluate_pdisc(
    results_genes=results_genes,
    adata_train=adata_train,
    adata_control=adata_control,
    pert_key=condition_keys
)

print(f"Global PDISC Mean: {pdisc_mean}")

In [ ]:
df_save_path = os.path.join(results_save_path, "metrics_detail.csv")
df_results.to_csv(df_save_path, index=True) 

In [ ]:
from src.evaluate import plot_perturbation_umap
plot_perturbation_umap(
    results_embedding=results_embedding,
    adata_control=adata_control,
    adata_conditions=adata_train, # 或者 adata_test
    target_genes=target_genes,
    control_indices=indices, 
    rep_key=sample_rep_scaled
)

In [ ]:
from src.evaluate import plot_perturbation_umap
plot_perturbation_umap(
    results_embedding=results_embedding,
    adata_control=adata_control,
    adata_conditions=adata_test, # 或者 adata_test
    target_genes=target_genes,
    control_indices=indices, 
    rep_key=sample_rep_scaled
)

In [ ]:
from src.evaluate import plot_top_degs_violin
for target in target_genes: # ['CDCA2', 'DHX36', 'TMSB10']
    plot_top_degs_violin(
        reconstructed_data=results_genes,
        adata_control=adata_control,
        adata_real=adata_train,
        target_gene=target,
        top_n=4 # 展示 Target + Top 4 下游基因
    )

In [ ]:
from src.evaluate import plot_top_degs_violin
for target in target_genes: # ['CDCA2', 'DHX36', 'TMSB10']
    plot_top_degs_violin(
        reconstructed_data=results_genes,
        adata_control=adata_control,
        adata_real=adata_test,
        target_gene=target,
        top_n=4 # 展示 Target + Top 4 下游基因
    )

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def diagnose_mass_prediction(pred_adata, gene_name='CDCA2'):
    mass = pred_adata.obs['mass'].values
    expr = pred_adata[:, gene_name].X.flatten() # 或者是 .layers['counts'] 等
    
    # 1. 打印统计数据
    print(f"=== Mass Statistics for {gene_name} prediction ===")
    print(f"Mean Mass: {np.mean(mass):.4f}")
    print(f"Max  Mass: {np.max(mass):.4f}")
    print(f"Min  Mass: {np.min(mass):.4f}")
    print(f"Std  Mass: {np.std(mass):.4f}")
    
    # 2. 绘图：Mass vs Target Gene Expression
    plt.figure(figsize=(10, 4))
    
    # 子图1: Mass 的直方图
    plt.subplot(1, 2, 1)
    plt.hist(mass, bins=50, color='green', alpha=0.7)
    plt.title("Distribution of Predicted Mass")
    plt.xlabel("Mass")
    plt.ylabel("Count")
    
    # 子图2: Mass vs Expression
    plt.subplot(1, 2, 2)
    plt.scatter(expr, mass, alpha=0.5, s=10, c=mass, cmap='viridis')
    plt.title(f"Mass vs {gene_name} Expression")
    plt.xlabel(f"{gene_name} Expression")
    plt.ylabel("Predicted Mass")
    plt.colorbar(label='Mass')
    
    plt.tight_layout()
    plt.show()

# 运行诊断
for gene in target_genes:
    diagnose_mass_prediction(results_genes[gene], gene)